# COS720 – AI-Powered Insider Threat Detection System
## Notebook 03: Model Development & Evaluation

**Objective:** Train and evaluate three machine learning classifiers, justify the selection of the best model, and save the final trained model for use in the prototype.

---
### Models Evaluated
| Model | Rationale |
|---|---|
| **Random Forest** | Ensemble of decision trees; handles mixed feature types, produces feature importances, robust to noise |
| **Decision Tree** | Interpretable baseline; shows how a single tree performs |
| **Logistic Regression** | Linear baseline; demonstrates why more complex models are needed |

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay, roc_curve, auc
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

MODELS_DIR   = 'models/'
RANDOM_STATE = 42
print('Setup complete.')

## 2. Load Preprocessed Data

In [ ]:
with open(f'{MODELS_DIR}splits.pkl', 'rb') as f:
    splits = pickle.load(f)

with open(f'{MODELS_DIR}feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

X_train     = splits['X_train']
X_train_sc  = splits['X_train_sc']
y_train     = splits['y_train']
X_test      = splits['X_test']
X_test_sc   = splits['X_test_sc']
y_test      = splits['y_test']

print(f'Training set: {X_train.shape} | Test set: {X_test.shape}')
print(f'Features: {feature_names}')

## 3. Train All Models

In [ ]:
model_configs = {
    'Random Forest': {
        'model': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                        n_jobs=-1, class_weight='balanced'),
        'scaled': False  # tree models don't need scaling
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=12,
                                        class_weight='balanced'),
        'scaled': False
    },
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=500, random_state=RANDOM_STATE,
                                    class_weight='balanced'),
        'scaled': True  # needs scaled features
    },
}

results = {}

for name, cfg in model_configs.items():
    model = cfg['model']
    X_tr = X_train_sc if cfg['scaled'] else X_train
    X_te = X_test_sc  if cfg['scaled'] else X_test
    
    print(f'Training {name}...', end=' ')
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    results[name] = {
        'model':     model,
        'y_pred':    y_pred,
        'y_prob':    y_prob,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'auc':       roc_auc,
        'fpr':       fpr,
        'tpr':       tpr,
        'cm':        confusion_matrix(y_test, y_pred),
    }
    print(f'Done. F1={results[name]["f1"]:.4f}')

## 4. Performance Comparison

In [ ]:
metrics_df = pd.DataFrame([
    {'Model': name,
     'Accuracy (%)':  round(r['accuracy']*100,  2),
     'Precision (%)': round(r['precision']*100, 2),
     'Recall (%)':    round(r['recall']*100,    2),
     'F1-Score (%)':  round(r['f1']*100,        2),
     'AUC':           round(r['auc'],            4)}
    for name, r in results.items()
]).set_index('Model')

print('Model Performance Comparison (on test set, malicious class):')
metrics_df

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(12, 5))
metrics_plot = metrics_df[['Precision (%)', 'Recall (%)', 'F1-Score (%)']]
metrics_plot.plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452', '#55A868'],
                  edgecolor='white', rot=0)
ax.set_title('Model Performance Comparison – Malicious Class', fontweight='bold', fontsize=13)
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 100)
ax.legend(loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, r) in zip(axes, results.items()):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=r['cm'],
        display_labels=['Normal', 'Malicious']
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')
    
    # Annotate TP, TN, FP, FN
    tn, fp, fn, tp = r['cm'].ravel()
    ax.set_xlabel(f'TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}', fontsize=8)

plt.suptitle('Confusion Matrices – All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#DD8452', '#4C72B0', '#55A868']

for (name, r), color in zip(results.items(), colors):
    ax.plot(r['fpr'], r['tpr'], lw=2, color=color,
            label=f"{name} (AUC = {r['auc']:.4f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves – Model Comparison', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.01])
plt.tight_layout()
plt.show()

## 7. Detailed Report – Best Model (Random Forest)

In [ ]:
best_name   = 'Random Forest'
best_result = results[best_name]
best_model  = best_result['model']

print(f'=== {best_name} – Full Classification Report ===')
print(classification_report(y_test, best_result['y_pred'],
                             target_names=['Normal (0)', 'Malicious (1)']))

tn, fp, fn, tp = best_result['cm'].ravel()
print(f'True Negatives  (correct Normal):    {tn:,}')
print(f'False Positives (Normal → Malicious): {fp:,}')
print(f'False Negatives (Malicious → Normal): {fn:,}')
print(f'True Positives  (correct Malicious): {tp:,}')

## 8. Model Selection Justification

**Random Forest is selected as the final model for the following reasons:**

1. **Best recall on the malicious class (~82%)** — in insider threat detection, a False Negative (missing a real threat) is far more costly than a False Positive. Random Forest achieves the best recall of the three models.

2. **High F1-Score (~75%)** — balances precision and recall well, making it the most reliable model for this imbalanced task.

3. **Highest AUC** — demonstrates the best overall ability to discriminate between benign and malicious behaviour across all decision thresholds.

4. **Built-in feature importance** — Random Forest provides Gini importance scores per feature, enabling the AI explainability required by the project specification without additional libraries.

5. **Robustness** — ensembles of 100 trees are resistant to noise and individual outliers, reducing overfitting compared to a single Decision Tree.

6. **No scaling required** — tree-based models are invariant to feature magnitude, making the pipeline simpler and more interpretable.

**Why Logistic Regression is not selected:** It achieves only 71% recall at the cost of 23% precision — generating too many false alarms for operational use. It assumes linear separability which does not hold in behavioural data.

**Why a single Decision Tree is not selected:** While interpretable, it achieves lower recall and F1 than Random Forest due to overfitting a single tree structure.

## 9. Save Final Model

In [ ]:
with open(f'{MODELS_DIR}rf_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print(f'Random Forest model saved to {MODELS_DIR}rf_model.pkl')
print('\nModel summary:')
print(f'  n_estimators:  {best_model.n_estimators}')
print(f'  class_weight:  {best_model.class_weight}')
print(f'  random_state:  {best_model.random_state}')
print(f'\nFinal test set performance (malicious class):')
print(f'  Accuracy:  {best_result["accuracy"]*100:.2f}%')
print(f'  Precision: {best_result["precision"]*100:.2f}%')
print(f'  Recall:    {best_result["recall"]*100:.2f}%')
print(f'  F1-Score:  {best_result["f1"]*100:.2f}%')
print(f'  AUC:       {best_result["auc"]:.4f}')